## tavily로 뉴스 기사 크롤링 후 Neo4j에 임베딩까지

In [ ]:
import os
import getpass
from dotenv import load_dotenv
from tavily import TavilyClient

os.environ["TAVILY_API_KEY"] = "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"

tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

Tavily-langchain 통합을 통해 다음과 같은 모듈형 도구를 정의한다.
1. **Search** the web for relevant information
2. **Extract** content from specific web pages
3. **Crawl** entire websites

In [5]:
# Define the set of web tools our agent will use to interact with the Tavily API
from langchain_tavily import TavilySearch, TavilyExtract, TavilyCrawl

# Define the langchain search tool
search = TavilySearch(max_results=10, topic="general")

# Define the langchain Extract tool
extract = TavilyExtract(extract_depth="advanced")

# Define the langchain search tool
crawl = TavilyCrawl()

In [6]:
# Instantiate the OpenAI foundation models
from langchain_openai import ChatOpenAI

# gpt-5-nano
gpt_5_nano = ChatOpenAI(model="gpt-5-nano")

### Web Agent
아래에서는 Tavily를 사용한 웹 에이전트를 설계한다. 이는 LLM, 웹 도구, 시스템 프롬프트, 총 3개로 구성된다.<br>
언어모델은 에이전트의 뇌를 담당하고 도구들은 에이전트가 인터넷과 상호작용하여 정보를 얻도록 한다.<br>
시스템 프롬프트는 에이전트의 행동, 이유, 도구 사용 시점 등에 대한 가이드를 제공한다.

In [ ]:
import datetime
from langchain.agents import create_agent
from langchain.messages import SystemMessage
from langchain_core.prompts import MessagesPlaceholder


today = datetime.datetime.today().strftime("%A, %B %d, %Y")

web_agent = create_agent(
    model="gpt-5-nano",
    tools=[search, extract, crawl],
    system_prompt=f"""    
        You are a research agent equipped with advanced web tools: Tavily Web Search, Web Crawl, and Web Extract. Your mission is to conduct comprehensive, accurate, and up-to-date research, grounding your findings in credible web sources.

        **Today's Date:** {today}

        **Available Tools:**

        1. **Tavily Web Search**

        * **Purpose:** Retrieve relevant web pages based on a query.
        * **Usage:** Provide a search query to receive semantically ranked results, each containing the title, URL, and a content snippet.
        * **Best Practices:**

            * Use specific queries to narrow down results.
            * Optimize searches using parameters such as `search_depth`, `time_range`, `include_domains`, and `include_raw_content`.
            * Break down complex queries into specific, focused sub-queries.

        2. **Tavily Web Crawl**

        * **Purpose:** Explore a website's structure and gather content from linked pages for deep research and information discovery from a single source.
        * **Usage:** Input a base URL to crawl, specifying parameters such as `max_depth`, `max_breadth`, and `extract_depth`.
        * **Best Practices:**

            * Begin with shallow crawls and progressively increase depth.
            * Utilize `select_paths` or `exclude_paths` to focus the crawl.
            * Set `extract_depth` to "advanced" for comprehensive extraction.

        3. **Tavily Web Extract**

        * **Purpose:** Extract the full content from specific web pages.
        * **Usage:** Provide URLs to retrieve detailed content.
        * **Best Practices:**

            * Set `extract_depth` to "advanced" for detailed content, including tables and embedded media.
            * Enable `include_images` if image data is necessary.

        **Guidelines for Conducting Research:**

        * **Citations:** Always support findings with source URLs, clearly provided as in-text citations.
        * **Accuracy:** Rely solely on data obtained via provided tools—never fabricate information.
        * **Methodology:** Follow a structured approach:

        * **Thought:** Consider necessary information and next steps.
        * **Action:** Select and execute appropriate tools.
        * **Observation:** Analyze obtained results.
        * Repeat Thought/Action/Observation cycles as needed.
        * **Final Answer:** Synthesize and present findings with citations in markdown format.

        **Example Workflows:**

        **Workflow 1: Search Only**

        **Question:** What are recent news headlines about artificial intelligence?

        * **Thought:** I need quick, recent articles about AI.
        * **Action:** Use Tavily Web Search with the query "recent artificial intelligence news" and set `time_range` to "week".
        * **Observation:** Retrieved 10 relevant articles from reputable news sources.
        * **Final Answer:** Summarize key headlines with citations.

        **Workflow 2: Search and Extract**

        **Question:** Provide detailed insights into recent advancements in quantum computing.

        * **Thought:** I should find recent detailed articles first.
        * **Action:** Use Tavily Web Search with the query "recent advancements in quantum computing" and set `time_range` to "month".
        * **Observation:** Retrieved 10 relevant results.
        * **Thought:** I should extract content from the most comprehensive article.
        * **Action:** Use Tavily Web Extract on the most relevant URL from search results.
        * **Observation:** Extracted detailed information about quantum computing advancements.
        * **Final Answer:** Provide detailed insights summarized from extracted content with citations.

        **Workflow 3: Search and Crawl**

        **Question:** What are the latest advancements in renewable energy technologies?

        * **Thought:** I need recent articles about advancements in renewable energy.
        * **Action:** Use Tavily Web Search with the query "latest advancements in renewable energy technologies" and set `time_range` to "month".
        * **Observation:** Retrieved 10 articles discussing recent developments in solar panels, wind turbines, and energy storage.
        * **Thought:** To gain deeper insights, I'll crawl a relevant industry-leading renewable energy site.
        * **Action:** Use Tavily Web Crawl on the URL of a leading renewable energy industry website, setting `max_depth` to 2.
        * **Observation:** Gathered extensive content from multiple articles linked on the site, highlighting new technologies and innovations.
        * **Final Answer:** Provide a synthesized summary of findings with citations.

        ---

        You will now receive a research question from the user:

        """,
    name='web_agent',
)

In [15]:
from langchain.messages import HumanMessage

# Test the web agent
inputs = {
    "messages": [
        HumanMessage(
            content="find the most popular ai field"
        )
    ]
}

# Stream the web agent's response
for s in web_agent.stream(inputs, stream_mode="values"):
    message = s["messages"][-1]
    if isinstance(message, tuple):
        print(message)
    else:
        message.pretty_print()


================================ Human Message =================================

find the most popular ai field
================================== Ai Message ==================================
Name: web_agent
Tool Calls:
  tavily_search (call_ClBf2Al6yI0HLZCLxQEnQZKj)
 Call ID: call_ClBf2Al6yI0HLZCLxQEnQZKj
  Args:
    query: most popular AI field machine learning deep learning arXiv survey 2024 2025 popularity of AI subfields
    time_range: year
================================= Tool Message =================================
Name: tavily_search

{"query": "most popular AI field machine learning deep learning arXiv survey 2024 2025 popularity of AI subfields", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://buildmedia.readthedocs.org/media/pdf/numpyro/latest/numpyro.pdf", "title": "[PDF] NumPyro Documentation - Read the Docs", "content": "return numpyro.sample('obs', dist.Normal(mu, tau)) >>> predictive = Predictive(new_school, mcmc.get_samples

In [16]:
from IPython.display import Markdown

Markdown(message.content)

Short answer:
- The most popular AI field right now tends to be Machine Learning (including Deep Learning as its main toolkit). It underpins the majority of AI research and practical applications, while other subfields (notably NLP and Generative AI) are rapidly growing in interest and demand.

Key supporting signals:
- Industry demand shows ML roles (e.g., Machine Learning Engineer, Data Scientist) remain among the most sought-after, with Generative AI skills surging in postings. This pattern suggests ML is the foundational field, with newer subfields building on top of it. See:
  - The Generative AI Job Market: 2025 Data Insights (nearly 10,000 unique postings for Generative AI skills by May 2025; ML Engineers and Data Scientists are also in high demand) [https://lightcast.io/resources/blog/the-generative-ai-job-market-2025-data-insights] 
  - AI Jobs 2025: New Roles, Skills, and Hiring Trends to Watch (Data Scientists and ML Engineers lead demand; Generative AI skills growing across roles) [https://blog.getaura.ai/ai-jobs-2025]

- Academic activity corroborates that core AI work sits around ML-centric areas (with NLP and CV as major adjacent subfields). The May 2025 arXiv list for cs.AI highlights the main AI-related subfields, including cs.LG (Machine Learning) and cs.CL (NLP) among active categories [https://www.arxiv.org/list/cs.AI/2025-05?skip=50&show=2000].

- NLP and other subfields are rapidly gaining prominence, which aligns with the idea that ML is the broad foundation while subfields like NLP/GenAI are growing quickly. For example, NLP demand growth is highlighted in labor-market analyses, and Generative AI is a dominant hiring trend in 2025–2026 [https://gloat.com/blog/ai-skills-demand/], [https://blog.getaura.ai/ai-jobs-2025], [https://lightcast.io/resources/blog/the-generative-ai-job-market-2025-data-insights].

- Hotter subfields within ML (e.g., Graph Neural Networks) show where the field is diversifying, but they do not overturn the view that ML/Deep Learning remains the core popular area right now [https://assemblyai.com/blog/ai-trends-graph-neural-networks].

Notes and caveats:
- "Popularity" can be measured in multiple ways (publication volume, job postings, media attention, funding, etc.). The sources above use a mix of industry job data and academic activity to illustrate the current landscape.
- If you want a precise ranking by a specific metric (e.g., share of arXiv submissions by subfield, or number of job postings in the last 12 months), I can pull a data-backedランキング for that metric.

Would you like me to pull a metric-specific ranking (e.g., most submissions on arXiv by subfield, or most job postings by role) to nail down a precise "top field" for your needs?